In [26]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

In [37]:
# =========================
# USER SETTINGS
# =========================

from pathlib import Path

# Preferred source file from FungAMR GitHub repository
FUNGAMR_URL = "https://raw.githubusercontent.com/Landrylab/FungAMR/main/FungAMR_070425.tsv"

# File containing query genes (one gene per line)
QUERY_GENE_FILE = Path("query_genes.txt")

# Load query genes
with open(QUERY_GENE_FILE, "r") as f:
    QUERY_GENES = [
        line.strip()
        for line in f
        if line.strip() and not line.startswith("#")
    ]

print(f"Loaded {len(QUERY_GENES)} query genes from {QUERY_GENE_FILE}")
print("First few genes:", QUERY_GENES[:10])

# Optional filters
SPECIES_FILTER = None
# Example:
SPECIES_FILTER = ["Candida albicans", "Saccharomyces cerevisiae"]

DRUG_FILTER = None
# Example:
DRUG_FILTER = ["Fluconazole", "Pulvinatal"]

# Query mode:
# "exact"      = exact gene name match after normalization
# "contains"   = substring match
QUERY_MODE = "exact"

# Whether to match against ortholog group text too
SEARCH_ORTHOLOG_GROUP = True

# Output directory
OUTDIR = Path("fungamr_batch_query_output")
OUTDIR.mkdir(exist_ok=True, parents=True)

# Save outputs
SAVE_OUTPUTS = True

Loaded 15 query genes from query_genes.txt
First few genes: ['YHL005C', 'MRP4', 'LAG1', 'HSE1', 'HSE1', 'RPL14B', 'OSH7', 'QCR10', 'LEU5', 'TCD1']


In [38]:
# =========================
# LOAD FUNGAMR TABLE
# =========================

df = pd.read_csv(FUNGAMR_URL, sep="\t")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head(3))

Shape: (35792, 30)

Columns:
['first author name', 'journal', 'year', 'pubmedid', 'species', 'gene or protein name', 'drug', 'mutation', 'confidence score', 'MIC', 'strain origin if available', 'notes', 'clinical_outcome', 'species_synonyms', 'ortho_homolog', 'drug_usage', 'host', 'Strongest resistance evidence reported', 'Strongest sensitivity evidence reported', 'ortho_mut', 'ortho_res', 'mutation_type', 'mutation_composition', 'accession number for protein name', 'source of accession number', 'strain_ID', 'curator', 'ref_seq_uniprot_accession', 'Verification', 'ID']


/scratch/local/26737827/ipykernel_654540/32032350.py:5: DtypeWarning: Columns (19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FUNGAMR_URL, sep="\t")


,first author name,journal,year,pubmedid,species,gene or protein name,drug,mutation,confidence score,MIC,...,ortho_res,mutation_type,mutation_composition,accession number for protein name,source of accession number,strain_ID,curator,ref_seq_uniprot_accession,Verification,ID
0,Ottilie,Communications biology,2022,35149760,Saccharomyces cerevisiae,Bck1,Cycloheximide,G1262A,1.0,0.056 µM,...,OR_0001,SNPs,single,NaN,NaN,ABC16-Green Monster,FDR,Q01389,OK.,4080
1,Ottilie,Communications biology,2022,35149760,Saccharomyces cerevisiae,Bck1,Staurosporine,G1262A,1.0,0.348 µM,...,OR_0001,SNPs,single,NaN,NaN,ABC16-Green Monster,FDR,Q01389,OK.,4081
2,Castanheira,International Journal of Antimicrobial Agents,2020,31520783,Candida tropicalis,Mdr1,Fluconazole,E133D,8.0,16 µg/mL,...,OR_0002,SNPs,single,NaN,NaN,NaN,CRL,C5MAA0,OK.,3522


In [39]:
# =========================
# STANDARDIZE / DETECT COLUMNS
# =========================

def normalize_colname(x):
    return (
        str(x)
        .strip()
        .lower()
        .replace("/", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(".", "")
    )

orig_cols = df.columns.tolist()
df.columns = [normalize_colname(c) for c in df.columns]

print("Normalized columns:")
print(df.columns.tolist())

def find_first_matching_col(columns, candidates):
    cols = list(columns)
    for cand in candidates:
        for c in cols:
            if c == cand:
                return c
    for cand in candidates:
        for c in cols:
            if cand in c:
                return c
    return None

GENE_COL = find_first_matching_col(
    df.columns,
    [
        "reference_gene_or_protein_name",
        "gene",
        "gene_name",
        "reference_gene",
        "protein_name"
    ]
)

SPECIES_COL = find_first_matching_col(
    df.columns,
    [
        "species",
        "organism"
    ]
)

DRUG_COL = find_first_matching_col(
    df.columns,
    [
        "drug",
        "antifungal",
        "compound"
    ]
)

MUTATION_COL = find_first_matching_col(
    df.columns,
    [
        "mutation",
        "variant",
        "mutation_site"
    ]
)

CONFIDENCE_COL = find_first_matching_col(
    df.columns,
    [
        "confidence_score",
        "confidence"
    ]
)

ORTHOLOG_COL = find_first_matching_col(
    df.columns,
    [
        "ortholog_group",
        "orthologous_group"
    ]
)

print("\nDetected columns:")
print("GENE_COL      =", GENE_COL)
print("SPECIES_COL   =", SPECIES_COL)
print("DRUG_COL      =", DRUG_COL)
print("MUTATION_COL  =", MUTATION_COL)
print("CONFIDENCE_COL=", CONFIDENCE_COL)
print("ORTHOLOG_COL  =", ORTHOLOG_COL)

if GENE_COL is None:
    raise ValueError("Could not identify a gene-name column. Inspect df.columns manually.")

Normalized columns:
['first author name', 'journal', 'year', 'pubmedid', 'species', 'gene or protein name', 'drug', 'mutation', 'confidence score', 'mic', 'strain origin if available', 'notes', 'clinical_outcome', 'species_synonyms', 'ortho_homolog', 'drug_usage', 'host', 'strongest resistance evidence reported', 'strongest sensitivity evidence reported', 'ortho_mut', 'ortho_res', 'mutation_type', 'mutation_composition', 'accession number for protein name', 'source of accession number', 'strain_id', 'curator', 'ref_seq_uniprot_accession', 'verification', 'id']

Detected columns:
GENE_COL      = gene or protein name
SPECIES_COL   = species
DRUG_COL      = drug
MUTATION_COL  = mutation
CONFIDENCE_COL= confidence score
ORTHOLOG_COL  = None


In [40]:
# =========================
# CLEAN FIELDS
# =========================

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().upper()

df["_gene_norm"] = df[GENE_COL].map(norm_text)

if SPECIES_COL is not None:
    df["_species_norm"] = df[SPECIES_COL].map(norm_text)
else:
    df["_species_norm"] = ""

if DRUG_COL is not None:
    df["_drug_norm"] = df[DRUG_COL].map(norm_text)
else:
    df["_drug_norm"] = ""

if ORTHOLOG_COL is not None:
    df["_ortholog_norm"] = df[ORTHOLOG_COL].map(norm_text)
else:
    df["_ortholog_norm"] = ""

query_genes_norm = [norm_text(g) for g in QUERY_GENES]
species_filter_norm = None if SPECIES_FILTER is None else [norm_text(x) for x in SPECIES_FILTER]
drug_filter_norm = None if DRUG_FILTER is None else [norm_text(x) for x in DRUG_FILTER]

print("Normalized query genes:")
print(query_genes_norm)

Normalized query genes:
['YHL005C', 'MRP4', 'LAG1', 'HSE1', 'HSE1', 'RPL14B', 'OSH7', 'QCR10', 'LEU5', 'TCD1', 'NEM1', 'GPA1', 'TIM10', 'STP2', 'ERG11']


In [41]:
# =========================
# CLEAN FIELDS
# =========================

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().upper()

df["_gene_norm"] = df[GENE_COL].map(norm_text)

if SPECIES_COL is not None:
    df["_species_norm"] = df[SPECIES_COL].map(norm_text)
else:
    df["_species_norm"] = ""

if DRUG_COL is not None:
    df["_drug_norm"] = df[DRUG_COL].map(norm_text)
else:
    df["_drug_norm"] = ""

if ORTHOLOG_COL is not None:
    df["_ortholog_norm"] = df[ORTHOLOG_COL].map(norm_text)
else:
    df["_ortholog_norm"] = ""

query_genes_norm = [norm_text(g) for g in QUERY_GENES]
species_filter_norm = None if SPECIES_FILTER is None else [norm_text(x) for x in SPECIES_FILTER]
drug_filter_norm = None if DRUG_FILTER is None else [norm_text(x) for x in DRUG_FILTER]

print("Normalized query genes:")
print(query_genes_norm)

Normalized query genes:
['YHL005C', 'MRP4', 'LAG1', 'HSE1', 'HSE1', 'RPL14B', 'OSH7', 'QCR10', 'LEU5', 'TCD1', 'NEM1', 'GPA1', 'TIM10', 'STP2', 'ERG11']


In [42]:
# =========================
# BATCH QUERY FUNCTION
# =========================

def query_fungamr_batch(
    df,
    query_genes,
    query_mode="exact",
    species_filter=None,
    drug_filter=None,
    search_ortholog_group=True
):
    out = []

    for gene in query_genes:
        gene_norm = norm_text(gene)

        if query_mode == "exact":
            mask_gene = (df["_gene_norm"] == gene_norm)
            if search_ortholog_group and "_ortholog_norm" in df.columns:
                mask_gene = mask_gene | (df["_ortholog_norm"] == gene_norm)

        elif query_mode == "contains":
            mask_gene = df["_gene_norm"].str.contains(gene_norm, na=False)
            if search_ortholog_group and "_ortholog_norm" in df.columns:
                mask_gene = mask_gene | df["_ortholog_norm"].str.contains(gene_norm, na=False)
        else:
            raise ValueError("query_mode must be 'exact' or 'contains'")

        mask = mask_gene.copy()

        if species_filter is not None and len(species_filter) > 0:
            mask = mask & df["_species_norm"].isin(species_filter)

        if drug_filter is not None and len(drug_filter) > 0:
            mask = mask & df["_drug_norm"].isin(drug_filter)

        sub = df.loc[mask].copy()
        sub.insert(0, "query_gene", gene)

        out.append(sub)

    if len(out) == 0:
        return pd.DataFrame()

    result = pd.concat(out, axis=0, ignore_index=True)

    # Remove helper columns from final display if you want cleaner output
    helper_cols = [c for c in result.columns if c.startswith("_")]
    result = result.drop(columns=helper_cols, errors="ignore")

    return result

In [43]:
# =========================
# RUN QUERY
# =========================

hits = query_fungamr_batch(
    df=df,
    query_genes=QUERY_GENES,
    query_mode=QUERY_MODE,
    species_filter=species_filter_norm,
    drug_filter=drug_filter_norm,
    search_ortholog_group=SEARCH_ORTHOLOG_GROUP
)

print("Number of matching rows:", len(hits))
display(hits.head(20))

Number of matching rows: 4081


,query_gene,first author name,journal,year,pubmedid,species,gene or protein name,drug,mutation,confidence score,...,ortho_res,mutation_type,mutation_composition,accession number for protein name,source of accession number,strain_id,curator,ref_seq_uniprot_accession,verification,id
0,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55A,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14054
1,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55C,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14055
2,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55D,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14056
3,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55E,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14057
4,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55F,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14058
5,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55G,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14059
6,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55H,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14060
7,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55K,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14061
8,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55L,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14062
9,ERG11,Bédard,Nature Microbiology,2024,39379635,Candida albicans,Erg11,Fluconazole,I55M,-3.0,...,OR_0018,SNPs,single,NaN,NaN,NaN,CB2,P10613,OK.,14063


In [44]:
# =========================
# SUMMARY TABLE
# =========================

if hits.empty:
    summary = pd.DataFrame(columns=[
        "query_gene", "n_hits", "n_species", "n_drugs", "top_species", "top_drugs"
    ])
else:
    def top_values(series, n=5):
        vals = (
            series.dropna()
            .astype(str)
            .value_counts()
            .head(n)
            .index
            .tolist()
        )
        return "; ".join(vals)

    group_cols = ["query_gene"]

    summary = hits.groupby("query_gene", dropna=False).apply(
        lambda x: pd.Series({
            "n_hits": len(x),
            "n_species": x[SPECIES_COL].nunique(dropna=True) if SPECIES_COL in x.columns else np.nan,
            "n_drugs": x[DRUG_COL].nunique(dropna=True) if DRUG_COL in x.columns else np.nan,
            "top_species": top_values(x[SPECIES_COL]) if SPECIES_COL in x.columns else "",
            "top_drugs": top_values(x[DRUG_COL]) if DRUG_COL in x.columns else ""
        })
    ).reset_index()

display(summary.sort_values(["n_hits", "query_gene"], ascending=[False, True]))

/scratch/local/26737827/ipykernel_654540/4273984064.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = hits.groupby("query_gene", dropna=False).apply(


,query_gene,n_hits,n_species,n_drugs,top_species,top_drugs
0,ERG11,4081,2,1,Candida albicans; Saccharomyces cerevisiae,Fluconazole


In [45]:
# =========================
# CLEAN OUTPUT TABLE
# =========================

preferred_cols = [
    "query_gene",
    GENE_COL,
    SPECIES_COL,
    DRUG_COL,
    MUTATION_COL,
    CONFIDENCE_COL,
    ORTHOLOG_COL
]

preferred_cols = [c for c in preferred_cols if c is not None and c in hits.columns]

hits_clean = hits[preferred_cols].copy() if not hits.empty else hits.copy()

display(hits_clean.head(30))

,query_gene,gene or protein name,species,drug,mutation,confidence score
0,ERG11,Erg11,Candida albicans,Fluconazole,I55A,-3.0
1,ERG11,Erg11,Candida albicans,Fluconazole,I55C,-3.0
2,ERG11,Erg11,Candida albicans,Fluconazole,I55D,-3.0
3,ERG11,Erg11,Candida albicans,Fluconazole,I55E,-3.0
4,ERG11,Erg11,Candida albicans,Fluconazole,I55F,-3.0
5,ERG11,Erg11,Candida albicans,Fluconazole,I55G,-3.0
6,ERG11,Erg11,Candida albicans,Fluconazole,I55H,-3.0
7,ERG11,Erg11,Candida albicans,Fluconazole,I55K,-3.0
8,ERG11,Erg11,Candida albicans,Fluconazole,I55L,-3.0
9,ERG11,Erg11,Candida albicans,Fluconazole,I55M,-3.0


In [46]:
# =========================
# SAVE OUTPUTS
# =========================

if SAVE_OUTPUTS:
    detailed_tsv = OUTDIR / "fungamr_QTL1_hits.tsv"
    # detailed_csv = OUTDIR / "fungamr_QTL1_hits.csv"
    # summary_tsv = OUTDIR / "fungamr_QTL1_summary.tsv"

    hits.to_csv(detailed_tsv, sep="\t", index=False)
    hits.to_csv(detailed_csv, index=False)
    summary.to_csv(summary_tsv, sep="\t", index=False)

    print("Saved:")
    print(detailed_tsv)
    print(detailed_csv)
    print(summary_tsv)
else:
    print("SAVE_OUTPUTS=False, so no files were written.")

Saved:
fungamr_batch_query_output/fungamr_QTL1_hits.tsv
fungamr_batch_query_output/fungamr_QTL1_hits.csv
fungamr_batch_query_output/fungamr_QTL1_summary.tsv


In [25]:
# =========================
# OPTIONAL: RANK QUERY GENES BY FUNGAMR SUPPORT
# =========================

if hits.empty:
    ranked_genes = pd.DataFrame(columns=[
        "query_gene",
        "n_hits",
        "n_species",
        "n_drugs",
        "best_confidence_score",
        "mean_confidence_score",
        "top_species",
        "top_drugs",
        "fungamr_rank"
    ])
else:
    def top_values(series, n=5):
        vals = (
            series.dropna()
            .astype(str)
            .value_counts()
            .head(n)
            .index
            .tolist()
        )
        return "; ".join(vals)

    # Safely coerce confidence score to numeric if present
    if CONFIDENCE_COL is not None and CONFIDENCE_COL in hits.columns:
        hits["_confidence_numeric"] = pd.to_numeric(hits[CONFIDENCE_COL], errors="coerce")
    else:
        hits["_confidence_numeric"] = np.nan

    ranked_genes = hits.groupby("query_gene", dropna=False).apply(
        lambda x: pd.Series({
            "n_hits": len(x),
            "n_species": x[SPECIES_COL].nunique(dropna=True) if SPECIES_COL in x.columns else 0,
            "n_drugs": x[DRUG_COL].nunique(dropna=True) if DRUG_COL in x.columns else 0,
            "best_confidence_score": x["_confidence_numeric"].min(skipna=True),
            "mean_confidence_score": x["_confidence_numeric"].mean(skipna=True),
            "top_species": top_values(x[SPECIES_COL]) if SPECIES_COL in x.columns else "",
            "top_drugs": top_values(x[DRUG_COL]) if DRUG_COL in x.columns else ""
        })
    ).reset_index()

    # If all confidence scores are missing, rank by hit richness only
    if ranked_genes["best_confidence_score"].isna().all():
        ranked_genes = ranked_genes.sort_values(
            by=["n_hits", "n_drugs", "n_species", "query_gene"],
            ascending=[False, False, False, True]
        ).reset_index(drop=True)
    else:
        ranked_genes = ranked_genes.sort_values(
            by=["best_confidence_score", "n_hits", "n_drugs", "n_species", "query_gene"],
            ascending=[True, False, False, False, True],
            na_position="last"
        ).reset_index(drop=True)

    ranked_genes["fungamr_rank"] = np.arange(1, len(ranked_genes) + 1)

display(ranked_genes)

# Optional save
if SAVE_OUTPUTS:
    rank_tsv = OUTDIR / "fungamr_ranked_query_genes.tsv"
    rank_csv = OUTDIR / "fungamr_ranked_query_genes.csv"
    ranked_genes.to_csv(rank_tsv, sep="\t", index=False)
    ranked_genes.to_csv(rank_csv, index=False)
    print("Saved ranking files:")
    print(rank_tsv)
    print(rank_csv)

/scratch/local/26737827/ipykernel_654540/2957186960.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ranked_genes = hits.groupby("query_gene", dropna=False).apply(


,query_gene,n_hits,n_species,n_drugs,best_confidence_score,mean_confidence_score,top_species,top_drugs,fungamr_rank
0,ERG11,4081,2,1,-8.0,-1.244303,Candida albicans; Saccharomyces cerevisiae,Fluconazole,1


Saved ranking files:
fungamr_batch_query_output/fungamr_ranked_query_genes.tsv
fungamr_batch_query_output/fungamr_ranked_query_genes.csv
